In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "gif"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 180

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "telemetry_orbital_mechanics"

CANVAS_SIZE = (1200, 720)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)


# =========================================================
# HELPERS
# =========================================================

def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be: webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def solve_kepler(mean_anomaly: float, eccentricity: float, iterations: int = 8) -> float:
    E = mean_anomaly
    for _ in range(iterations):
        E -= (E - eccentricity * np.sin(E) - mean_anomaly) / (1 - eccentricity * np.cos(E))
    return E


def orbit_point(a: float, e: float, mean_anomaly: float) -> tuple[float, float, float]:
    E = solve_kepler(mean_anomaly, e)
    x = a * (np.cos(E) - e)
    y = a * np.sqrt(1 - e * e) * np.sin(E)
    r = np.sqrt(x * x + y * y)
    return x, y, r


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 5):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


# =========================================================
# DRAW
# =========================================================

def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 34
    cut = 52

    pts = [
        (pad + cut, pad),
        (W - pad, pad),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 22, pad + 22, W - pad - 22, H - pad - 22],
        outline=(*CYAN2, 42),
        width=1,
    )

    d.text(
        (pad + 24, pad + 14),
        "TELEMETRY // ORBITAL MECHANICS",
        font=font,
        fill=(*CYAN2, 220),
    )


def draw_grid(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 84, x, H - 84], fill=(*CYAN, 18), width=1)

    for y in range(100, H - 80, 60):
        d.line([84, y, W - 84, y], fill=(*CYAN, 14), width=1)


def draw_star(layer: Image.Image, cx: float, cy: float, phase: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.75 + 0.25 * np.sin(phase * 2) ** 2

    for r, a in [(52, 26), (36, 52), (22, 120)]:
        d.ellipse(
            [cx - r, cy - r, cx + r, cy + r],
            fill=(*YELLOW, int(a * pulse)),
        )

    d.ellipse(
        [cx - 13, cy - 13, cx + 13, cy + 13],
        fill=(*WHITE, 230),
        outline=(*YELLOW, 240),
        width=2,
    )


def draw_planet(layer: Image.Image, x: float, y: float, phase: float):
    d = ImageDraw.Draw(layer)

    pulse = 0.65 + 0.35 * np.sin(phase * 5) ** 2

    d.ellipse(
        [x - 28, y - 28, x + 28, y + 28],
        outline=(*CYAN2, int(115 * pulse)),
        width=2,
    )

    d.ellipse(
        [x - 10, y - 10, x + 10, y + 10],
        fill=(*CYAN2, 230),
    )

    d.ellipse(
        [x - 4, y - 4, x + 4, y + 4],
        fill=(*WHITE, 235),
    )


def draw_orbit_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)
    font_big = load_font(22)
    font_small = load_font(15)
    font_mid = load_font(18)

    W, H = frame.size
    cx, cy = W * 0.70, H * 0.52

    phase = 2 * np.pi * i / TOTAL_FRAMES

    draw_hud_frame(frame, font_big)
    draw_grid(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(scene)




    # orbital parameters
    a = 260.0
    e = 0.48
    b = a * np.sqrt(1 - e * e)

    focus_x = cx - a * e
    focus_y = cy

    # orbit path — ellipse with star in focus
    orbit_pts = []

    for t in np.linspace(0, 2 * np.pi, 720):
        x = focus_x + a * (np.cos(t) - e)
        y = focus_y + b * np.sin(t)
        orbit_pts.append((x, y))

    d.line(orbit_pts + [orbit_pts[0]], fill=(*CYAN2, 165), width=2)

    # geometric center of ellipse
    center_x = focus_x - a * e
    center_y = focus_y

    # dashed reference major/minor axes
    d.line([center_x - a, center_y, center_x + a, center_y], fill=(*CYAN, 50), width=1)
    d.line([center_x, center_y - b, center_x, center_y + b], fill=(*CYAN, 34), width=1)

    # apsides
    peri = (focus_x + a * (1 - e), focus_y)
    apo = (focus_x - a * (1 + e), focus_y)


    d.ellipse([peri[0] - 5, peri[1] - 5, peri[0] + 5, peri[1] + 5], fill=(*ORANGE, 220))
    d.ellipse([apo[0] - 5, apo[1] - 5, apo[0] + 5, apo[1] + 5], fill=(*GREEN, 200))

    d.text((peri[0] - 42, peri[1] + 16), "PERI", font=font_small, fill=(*ORANGE, 190))
    d.text((apo[0] + 12, apo[1] + 16), "APO", font=font_small, fill=(*GREEN, 180))

    # focus / star
    d.ellipse(
        [focus_x - 5, focus_y - 5, focus_x + 5, focus_y + 5],
        fill=(*YELLOW, 210),
    )

    draw_star(scene, focus_x, focus_y, phase)

    # planet position with Kepler motion
    mean_anomaly = phase
    ox, oy, r_phys = orbit_point(a, e, mean_anomaly)

    planet_x = focus_x + ox
    planet_y = focus_y + oy

    # radius vector
    d.line(
        [focus_x, focus_y, planet_x, planet_y],
        fill=(*YELLOW, 155),
        width=2,
    )

    # swept area triangle
    d.polygon(
        [
            (focus_x, focus_y),
            (planet_x, planet_y),
            (focus_x + a * 0.20 * np.cos(phase - 0.16), focus_y + b * 0.20 * np.sin(phase - 0.16)),
        ],
        fill=(*YELLOW, 26),
    )

    draw_planet(scene, planet_x, planet_y, phase)

    # velocity tangent hint
    ox2, oy2, _ = orbit_point(a, e, mean_anomaly + 0.025)
    px2 = focus_x + ox2
    py2 = focus_y + oy2
    vx = px2 - planet_x
    vy = py2 - planet_y
    vn = max(1e-6, np.sqrt(vx * vx + vy * vy))
    vx /= vn
    vy /= vn

    d.line(
        [planet_x, planet_y, planet_x + vx * 56, planet_y + vy * 56],
        fill=(*GREEN, 210),
        width=2,
    )

    d.text(
        (planet_x + vx * 62, planet_y + vy * 62),
        "v",
        font=font_mid,
        fill=(*GREEN, 220),
    )

    # true anomaly arc
    arc_r = 78
    d.arc(
        [focus_x - arc_r, focus_y - arc_r, focus_x + arc_r, focus_y + arc_r],
        start=0,
        end=np.degrees(np.arctan2(planet_y - focus_y, planet_x - focus_x)),
        fill=(*CYAN2, 130),
        width=2,
    )

    # labels
    d.text((82, 112), "ELLIPTICAL ORBIT / KEPLERIAN MOTION", font=font_mid, fill=(*CYAN2, 210))
    d.text((82, 144), f"eccentricity e = {e:.2f}", font=font_small, fill=(*CYAN2, 180))
    d.text((82, 168), "STAR AT FOCUS F1", font=font_small, fill=(*YELLOW, 185))
    d.text((82, 192), "RADIUS VECTOR + TANGENTIAL VELOCITY", font=font_small, fill=(*GREEN, 175))

    # telemetry box
    box_x0, box_y0 = W - 365, 108
    box_x1, box_y1 = W - 86, 250

    d.rectangle([box_x0, box_y0, box_x1, box_y1], fill=(*BG_DARK, 145))
    d.rectangle([box_x0, box_y0, box_x1, box_y1], outline=(*CYAN, 90), width=1)

    speed_proxy = 1.0 / max(0.35, r_phys / a)

    rows = [
        ("a", f"{a / 100:.2f} AU"),
        ("e", f"{e:.2f}"),
        ("r", f"{r_phys / 100:.2f} AU"),
        ("v", f"{speed_proxy:.2f} rel"),
    ]

    yy = box_y0 + 18
    for k, v in rows:
        d.text((box_x0 + 18, yy), k, font=font_small, fill=(*CYAN, 150))
        d.text((box_x0 + 94, yy), v, font=font_small, fill=(*CYAN2, 210))
        yy += 28

    # progress-like anomaly ring, but not a bottom runner
    ring_cx, ring_cy = W - 132, H - 118
    ring_r = 46
    d.ellipse(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        outline=(*CYAN, 70),
        width=1,
    )
    d.arc(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        start=-90,
        end=-90 + 360 * i / TOTAL_FRAMES,
        fill=(*CYAN2, 220),
        width=4,
    )
    d.text((ring_cx - 38, ring_cy + 58), "MEAN ANOMALY", font=font_small, fill=(*CYAN2, 150))

    glow_composite(frame, scene, blur=5)

    return frame


# =========================================================
# EXPORT
# =========================================================

def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


# =========================================================
# MAIN
# =========================================================

def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] orbital mechanics telemetry")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_orbit_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(
        f"Created 1 animation(s) as '{output_format}' "
        f"in '{ANIMATIONS_DIR.resolve()}'"
    )


main()

[START] orbital mechanics telemetry
[CONFIG] OUTPUT_FORMAT = gif
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180

[CREATED] telemetry_orbital_mechanics
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_orbital_mechanics/telemetry_orbital_mechanics.gif

Created 1 animation(s) as 'gif' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


In [8]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 180

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "telemetry_exoplanet_transit"

CANVAS_SIZE = (1200, 620)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
WHITE = (245, 250, 255)


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be: webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 5):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 34
    cut = 52

    pts = [
        (pad + cut, pad),
        (W - pad, pad),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 22, pad + 22, W - pad - 22, H - pad - 22],
        outline=(*CYAN2, 42),
        width=1,
    )

    d.text(
        (pad + 24, pad + 14),
        "TELEMETRY // EXOPLANET TRANSIT",
        font=font,
        fill=(*CYAN2, 220),
    )


def draw_grid(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 86, x, H - 70], fill=(*CYAN, 14), width=1)

    for y in range(100, H - 70, 60):
        d.line([84, y, W - 84, y], fill=(*CYAN, 12), width=1)


def overlap_area_two_circles(d: float, r1: float, r2: float) -> float:
    if d >= r1 + r2:
        return 0.0

    if d <= abs(r1 - r2):
        return np.pi * min(r1, r2) ** 2

    part1 = r1 * r1 * np.arccos((d*d + r1*r1 - r2*r2) / (2*d*r1))
    part2 = r2 * r2 * np.arccos((d*d + r2*r2 - r1*r1) / (2*d*r2))
    part3 = 0.5 * np.sqrt(
        max(
            0.0,
            (-d + r1 + r2)
            * (d + r1 - r2)
            * (d - r1 + r2)
            * (d + r1 + r2),
        )
    )

    return part1 + part2 - part3


def draw_star_disc(layer: Image.Image, cx: float, cy: float, r: float):
    d = ImageDraw.Draw(layer)

    # limb-darkened star
    for k in range(80, 0, -1):
        rr = r * k / 80
        t = k / 80

        alpha = int(28 + 190 * (1 - 0.55 * (1 - t) ** 1.8))
        col = (
            int(255),
            int(220 + 22 * t),
            int(130 + 80 * t),
        )

        d.ellipse(
            [cx - rr, cy - rr, cx + rr, cy + rr],
            fill=(*col, alpha),
        )

    d.ellipse(
        [cx - r, cy - r, cx + r, cy + r],
        outline=(*YELLOW, 180),
        width=2,
    )

    # subtle equatorial scan lines
    for yy in np.linspace(cy - r * 0.75, cy + r * 0.75, 9):
        half = np.sqrt(max(0, r*r - (yy - cy) ** 2))
        d.line(
            [cx - half, yy, cx + half, yy],
            fill=(*YELLOW, 24),
            width=1,
        )


def draw_planet(layer: Image.Image, x: float, y: float, r: float):
    d = ImageDraw.Draw(layer)

    d.ellipse(
        [x - r * 1.45, y - r * 1.45, x + r * 1.45, y + r * 1.45],
        outline=(*CYAN2, 70),
        width=1,
    )

    d.ellipse(
        [x - r, y - r, x + r, y + r],
        fill=(1, 4, 8, 245),
        outline=(*CYAN2, 210),
        width=2,
    )


def draw_light_curve(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    progress: float,
    star_r_norm: float,
    planet_r_norm: float,
    impact: float,
    font,
):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 150))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    for gx in np.linspace(x0 + 20, x1 - 20, 9):
        d.line([gx, y0 + 12, gx, y1 - 18], fill=(*CYAN, 28), width=1)

    for gy in np.linspace(y0 + 16, y1 - 24, 5):
        d.line([x0 + 14, gy, x1 - 14, gy], fill=(*CYAN, 24), width=1)

    xs = np.linspace(0, 1, 420)
    flux = []

    for t in xs:
        px = -1.65 + 3.3 * t
        py = impact
        sep = np.sqrt(px * px + py * py)

        overlap = overlap_area_two_circles(
            sep,
            star_r_norm,
            planet_r_norm,
        )

        depth = overlap / (np.pi * star_r_norm * star_r_norm)
        flux.append(1.0 - depth)

    flux = np.array(flux)

    y_top = y0 + 24
    y_bottom = y1 - 34
    f_min = 1.0 - (planet_r_norm / star_r_norm) ** 2 * 1.15
    f_max = 1.002

    pts = []

    for t, f in zip(xs, flux):
        x = x0 + 18 + t * (x1 - x0 - 36)
        y = y_bottom - (f - f_min) / (f_max - f_min) * (y_bottom - y_top)
        pts.append((x, y))

    d.line(pts, fill=(*CYAN2, 230), width=2)

    # moving point
    idx = min(len(pts) - 1, max(0, int(progress * (len(pts) - 1))))
    mx, my = pts[idx]

    d.ellipse([mx - 6, my - 6, mx + 6, my + 6], fill=(*WHITE, 230), outline=(*GREEN, 230), width=1)

    d.text((x0 + 18, y0 + 12), "LIGHT CURVE", font=font, fill=(*CYAN2, 180))
    d.text((x1 - 138, y0 + 12), "FLUX", font=font, fill=(*GREEN, 160))


def draw_telemetry_box(layer: Image.Image, box: tuple[int, int, int, int], rows: list[tuple[str, str]], font):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 145))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    y = y0 + 18

    for key, value in rows:
        d.text((x0 + 18, y), key, font=font, fill=(*CYAN, 150))
        d.text((x0 + 145, y), value, font=font, fill=(*CYAN2, 215))
        y += 30


def draw_transit_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(22)
    font_small = load_font(15)
    font_mid = load_font(18)

    W, H = frame.size
    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = i / TOTAL_FRAMES

    draw_hud_frame(frame, font_big)
    draw_grid(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(scene)

    star_cx = W * 0.50
    star_cy = H * 0.39
    star_r = 128

    planet_r = 24
    impact = 0.32

    # one clean loop: planet starts right outside, crosses disk, exits
    track_x0 = star_cx - star_r * 1.75
    track_x1 = star_cx + star_r * 1.75
    planet_x = track_x0 + (track_x1 - track_x0) * progress
    planet_y = star_cy + impact * star_r

    # orbital/transit chord
    d.line(
        [track_x0, planet_y, track_x1, planet_y],
        fill=(*CYAN, 70),
        width=1,
    )

    draw_star_disc(scene, star_cx, star_cy, star_r)
    draw_planet(scene, planet_x, planet_y, planet_r)

    # contact markers
    for offset, label in [
        (-star_r - planet_r, "I"),
        (-star_r + planet_r, "II"),
        (star_r - planet_r, "III"),
        (star_r + planet_r, "IV"),
    ]:
        x = star_cx + offset
        d.line([x, planet_y - 18, x, planet_y + 18], fill=(*GREEN, 130), width=1)
        d.text((x - 6, planet_y + 24), label, font=font_small, fill=(*GREEN, 170))

    # projected orbit guide
    guide_y = H * 0.18
    d.arc(
        [star_cx - 260, guide_y - 70, star_cx + 260, guide_y + 70],
        0,
        360,
        fill=(*CYAN, 45),
        width=1,
    )

    d.ellipse(
        [planet_x - 4, guide_y - 4, planet_x + 4, guide_y + 4],
        fill=(*CYAN2, 180),
    )

    # instantaneous flux from geometric overlap
    px_norm = (planet_x - star_cx) / star_r
    py_norm = impact
    sep = np.sqrt(px_norm * px_norm + py_norm * py_norm)
    overlap = overlap_area_two_circles(sep, 1.0, planet_r / star_r)
    depth = overlap / np.pi
    flux = 1.0 - depth

    rows = [
        ("Rp/R*", f"{planet_r / star_r:.3f}"),
        ("impact", f"{impact:.2f}"),
        ("depth", f"{(1 - flux) * 100:.2f}%"),
        ("flux", f"{flux:.5f}"),
    ]

    draw_telemetry_box(
        scene,
        (W - 365, 132, W - 84, 276),
        rows,
        font_small,
    )

    d.text((82, 112), "PLANETARY TRANSIT GEOMETRY", font=font_mid, fill=(*CYAN2, 210))
    d.text((82, 140), "SYNCHRONIZED DISC OCCULTATION + LIGHT CURVE", font=font_small, fill=(*CYAN2, 160))

    draw_light_curve(
        scene,
        (90, H - 210, W - 90, H - 84),
        progress,
        star_r_norm=1.0,
        planet_r_norm=planet_r / star_r,
        impact=impact,
        font=font_small,
    )

    glow_composite(frame, scene, blur=5)

    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] exoplanet transit telemetry")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_transit_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(
        f"Created 1 animation(s) as '{output_format}' "
        f"in '{ANIMATIONS_DIR.resolve()}'"
    )


main()

[START] exoplanet transit telemetry
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] telemetry_exoplanet_transit
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_exoplanet_transit/telemetry_exoplanet_transit.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[out#0/webm @ 0x11ff24d00] video:126KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 68.178687%
frame=  180 fps= 47 q=34.0 Lsize=     212KiB time=00:00:07.50 bitrate= 231.7kbits/s speed=1.95x    


In [13]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 180

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "telemetry_binary_star_barycenter"

CANVAS_SIZE = (1200, 680)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be: webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 5):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 34
    cut = 52

    pts = [
        (pad + cut, pad),
        (W - pad, pad),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)
    d.rectangle(
        [pad + 22, pad + 22, W - pad - 22, H - pad - 22],
        outline=(*CYAN2, 42),
        width=1,
    )


    d.text(
        (pad + 24, pad + 14),
        "TELEMETRY // BINARY STAR BARYCENTER",
        font=font,
        fill=(*CYAN2, 220),
    )


def draw_grid(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 86, x, H - 70], fill=(*CYAN, 13), width=1)

    for y in range(100, H - 70, 60):
        d.line([84, y, W - 84, y], fill=(*CYAN, 11), width=1)


def draw_star(layer: Image.Image, x: float, y: float, r: float, color, phase: float, label: str, font):
    d = ImageDraw.Draw(layer)

    pulse = 0.72 + 0.28 * np.sin(phase * 3.0) ** 2

    for rr, a in [
        (r * 3.0, 6),
        (r * 2.1, 14),
        (r * 1.35, 34),
    ]:

        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            fill=(*color, int(a * pulse * 0.55)),
        )

    d.ellipse(
        [x - r, y - r, x + r, y + r],
        fill=(*color, 225),
        outline=(*WHITE, 220),
        width=2,
    )

    d.text((x + r + 12, y - 8), label, font=font, fill=(*color, 210))


def draw_dashed_ellipse(
    d: ImageDraw.ImageDraw,
    cx: float,
    cy: float,
    rx: float,
    ry: float,
    color,
    alpha: int,
    width: int = 1,
    dash: int = 10,
):
    pts = []
    n = 360

    for i in range(n + 1):
        t = 2 * np.pi * i / n
        pts.append((cx + rx * np.cos(t), cy + ry * np.sin(t)))

    for i in range(0, n, dash * 2):
        seg = pts[i:i + dash]
        if len(seg) > 1:
            d.line(seg, fill=(*color, alpha), width=width)


def radial_velocity_curve(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    phase: float,
    font,
):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 150))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    for gx in np.linspace(x0 + 20, x1 - 20, 8):
        d.line([gx, y0 + 14, gx, y1 - 18], fill=(*CYAN, 26), width=1)

    for gy in np.linspace(y0 + 16, y1 - 24, 5):
        d.line([x0 + 14, gy, x1 - 14, gy], fill=(*CYAN, 22), width=1)

    xs = np.linspace(0, 1, 360)
    mid_y = (y0 + y1) / 2
    amp = (y1 - y0) * 0.30

    pts1 = []
    pts2 = []

    for t in xs:
        x = x0 + 18 + t * (x1 - x0 - 36)
        y1v = mid_y - amp * np.sin(2 * np.pi * t)
        y2v = mid_y + amp * 0.62 * np.sin(2 * np.pi * t)
        pts1.append((x, y1v))
        pts2.append((x, y2v))

    d.line(pts1, fill=(*YELLOW, 220), width=2)
    d.line(pts2, fill=(*CYAN2, 220), width=2)

    mx = x0 + 18 + ((phase / (2 * np.pi)) % 1.0) * (x1 - x0 - 36)
    d.line([mx, y0 + 12, mx, y1 - 18], fill=(*GREEN, 150), width=1)

    d.text((x0 + 18, y0 + 12), "RADIAL VELOCITY", font=font, fill=(*CYAN2, 175))
    d.text((x1 - 160, y0 + 12), "PHASE LOCKED", font=font, fill=(*GREEN, 150))


def draw_binary_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(22)
    font_small = load_font(15)
    font_mid = load_font(18)

    W, H = frame.size
    phase = 2 * np.pi * i / TOTAL_FRAMES

    draw_hud_frame(frame, font_big)
    draw_grid(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(scene)

    # main coordinate center / barycenter
    cx = W * 0.46
    cy = H * 0.42

    # masses
    m1 = 1.45
    m2 = 0.82
    total_m = m1 + m2

    separation = 330.0

    # distance from barycenter: heavier star closer
    r1 = separation * m2 / total_m
    r2 = separation * m1 / total_m

    # projected orbital ellipse
    orbit_squash = 0.58

    x1 = cx + r1 * np.cos(phase + np.pi)
    y1 = cy + r1 * orbit_squash * np.sin(phase + np.pi)

    x2 = cx + r2 * np.cos(phase)
    y2 = cy + r2 * orbit_squash * np.sin(phase)

    # orbit paths
    draw_dashed_ellipse(d, cx, cy, r1, r1 * orbit_squash, YELLOW, 90, width=1)
    draw_dashed_ellipse(d, cx, cy, r2, r2 * orbit_squash, CYAN2, 90, width=1)

    # connecting line
    d.line([x1, y1, x2, y2], fill=(*CYAN, 80), width=1)

  
    # stars
    draw_star(scene, x1, y1, 28, YELLOW, phase, "M1", font_small)
    draw_star(scene, x2, y2, 18, CYAN2, phase + np.pi, "M2", font_small)
            # barycenter marker
    d.line([cx - 14, cy, cx + 14, cy], fill=(*GREEN, 180), width=1)
    d.line([cx, cy - 14, cx, cy + 14], fill=(*GREEN, 180), width=1)
    d.ellipse([cx - 5, cy - 5, cx + 5, cy + 5], fill=(*GREEN, 220))
    
    d.text((cx + 16, cy - 10), "BARYCENTER", font=font_small, fill=(*GREEN, 190))

    # direction vector arrows
    vx1 = -np.sin(phase + np.pi)
    vy1 = np.cos(phase + np.pi) * orbit_squash
    vx2 = -np.sin(phase)
    vy2 = np.cos(phase) * orbit_squash

    d.line([x1, y1, x1 + vx1 * 46, y1 + vy1 * 46], fill=(*YELLOW, 180), width=2)
    d.line([x2, y2, x2 + vx2 * 56, y2 + vy2 * 56], fill=(*CYAN2, 180), width=2)

    # title / description
    d.text((82, 112), "TWO-BODY SYSTEM / COMMON CENTER OF MASS", font=font_mid, fill=(*CYAN2, 210))
    d.text((82, 140), "MASSIVE STAR MOVES ON SMALLER ORBIT AROUND BARYCENTER", font=font_small, fill=(*CYAN2, 155))

    # telemetry box
    box_x0, box_y0 = W - 365, 112
    box_x1, box_y1 = W - 84, 286

    d.rectangle([box_x0, box_y0, box_x1, box_y1], fill=(*BG_DARK, 150))
    d.rectangle([box_x0, box_y0, box_x1, box_y1], outline=(*CYAN, 90), width=1)

    rows = [
        ("M1", f"{m1:.2f} Msun"),
        ("M2", f"{m2:.2f} Msun"),
        ("q", f"{m2 / m1:.2f}"),
        ("sep", f"{separation / 100:.2f} AU"),
        ("phase", f"{(i / TOTAL_FRAMES):.3f}"),
    ]

    yy = box_y0 + 18
    for key, value in rows:
        d.text((box_x0 + 18, yy), key, font=font_small, fill=(*CYAN, 150))
        d.text((box_x0 + 105, yy), value, font=font_small, fill=(*CYAN2, 215))
        yy += 29

    # radial velocity graph
    radial_velocity_curve(
        scene,
        (90, H - 210, W - 90, H - 82),
        phase,
        font_small,
    )

    # small phase ring
    ring_cx, ring_cy = W - 120, H - 298
    ring_r = 42
    d.ellipse(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        outline=(*CYAN, 70),
        width=1,
    )
    d.arc(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        start=-90,
        end=-90 + 360 * i / TOTAL_FRAMES,
        fill=(*CYAN2, 220),
        width=4,
    )
    d.text((ring_cx - 40, ring_cy + 54), "ORBIT PHASE", font=font_small, fill=(*CYAN2, 145))

    glow_composite(frame, scene, blur=5)

    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] binary star barycenter telemetry")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_binary_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(
        f"Created 1 animation(s) as '{output_format}' "
        f"in '{ANIMATIONS_DIR.resolve()}'"
    )


main()

[START] binary star barycenter telemetry
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] telemetry_binary_star_barycenter
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_binary_star_barycenter/telemetry_binary_star_barycenter.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


In [14]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 180

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "telemetry_pulsar_beam"

CANVAS_SIZE = (1200, 680)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)


def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be: webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 5):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 34
    cut = 52

    pts = [
        (pad + cut, pad),
        (W - pad, pad),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)
    d.rectangle(
        [pad + 22, pad + 22, W - pad - 22, H - pad - 22],
        outline=(*CYAN2, 42),
        width=1,
    )
    d.text(
        (pad + 24, pad + 14),
        "TELEMETRY // PULSAR BEAM GEOMETRY",
        font=font,
        fill=(*CYAN2, 220),
    )


def draw_grid(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 86, x, H - 70], fill=(*CYAN, 13), width=1)

    for y in range(100, H - 70, 60):
        d.line([84, y, W - 84, y], fill=(*CYAN, 11), width=1)


def ellipse_points(cx, cy, rx, ry, n=240, start=0, stop=2 * np.pi):
    pts = []
    for t in np.linspace(start, stop, n):
        pts.append((cx + rx * np.cos(t), cy + ry * np.sin(t)))
    return pts


def draw_neutron_star(layer: Image.Image, cx: float, cy: float, r: float, phase: float):
    d = ImageDraw.Draw(layer)

    # outer magnetosphere rings
    for idx, scale in enumerate([2.7, 2.15, 1.62]):
        rr = r * scale
        alpha = int(26 + 18 * np.sin(phase * 2 + idx) ** 2)
        d.ellipse(
            [cx - rr, cy - rr * 0.62, cx + rr, cy + rr * 0.62],
            outline=(*CYAN2, alpha),
            width=1,
        )

    # star glow
    for rr, alpha in [(r * 2.0, 18), (r * 1.45, 42), (r * 1.08, 90)]:
        d.ellipse(
            [cx - rr, cy - rr, cx + rr, cy + rr],
            fill=(*CYAN2, alpha),
        )

    # surface
    d.ellipse(
        [cx - r, cy - r, cx + r, cy + r],
        fill=(*CYAN2, 215),
        outline=(*WHITE, 210),
        width=2,
    )

    # rotating surface bands
    for k in range(5):
        yoff = (k - 2) * r * 0.28
        wobble = np.sin(phase * 2 + k) * 5
        d.arc(
            [cx - r * 0.9, cy + yoff - r * 0.22 + wobble,
             cx + r * 0.9, cy + yoff + r * 0.22 + wobble],
            0,
            360,
            fill=(*BG_DARK, 75),
            width=1,
        )


def draw_beam_cone(
    layer: Image.Image,
    origin: tuple[float, float],
    angle: float,
    length: float,
    spread: float,
    color: tuple[int, int, int],
    alpha: int,
):
    d = ImageDraw.Draw(layer)
    ox, oy = origin

    a1 = angle - spread
    a2 = angle + spread

    p1 = (ox + length * np.cos(a1), oy + length * np.sin(a1))
    p2 = (ox + length * np.cos(a2), oy + length * np.sin(a2))
    pc = (ox + length * np.cos(angle), oy + length * np.sin(angle))

    d.polygon(
        [(ox, oy), p1, pc, p2],
        fill=(*color, alpha),
        outline=(*color, min(210, alpha + 70)),
    )

    d.line([origin, pc], fill=(*WHITE, min(210, alpha + 60)), width=1)


def angular_distance(a: float, b: float) -> float:
    return abs(np.arctan2(np.sin(a - b), np.cos(a - b)))


def pulse_intensity(phase: float, observer_angle: float) -> float:
    beam1 = phase
    beam2 = phase + np.pi

    sigma = 0.13

    d1 = angular_distance(beam1, observer_angle)
    d2 = angular_distance(beam2, observer_angle)

    return float(
        np.exp(-0.5 * (d1 / sigma) ** 2)
        + 0.82 * np.exp(-0.5 * (d2 / sigma) ** 2)
    )


def draw_pulse_profile(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    phase: float,
    observer_angle: float,
    font,
):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 150))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    for gx in np.linspace(x0 + 20, x1 - 20, 8):
        d.line([gx, y0 + 14, gx, y1 - 18], fill=(*CYAN, 26), width=1)

    for gy in np.linspace(y0 + 16, y1 - 24, 5):
        d.line([x0 + 14, gy, x1 - 14, gy], fill=(*CYAN, 22), width=1)

    xs = np.linspace(0, 1, 480)
    pts = []

    y_top = y0 + 24
    y_bottom = y1 - 30

    for t in xs:
        ph = 2 * np.pi * t
        val = min(1.0, pulse_intensity(ph, observer_angle))
        x = x0 + 18 + t * (x1 - x0 - 36)
        y = y_bottom - val * (y_bottom - y_top)
        pts.append((x, y))

    d.line(pts, fill=(*CYAN2, 230), width=2)

    current = (phase / (2 * np.pi)) % 1.0
    mx = x0 + 18 + current * (x1 - x0 - 36)
    val = min(1.0, pulse_intensity(phase, observer_angle))
    my = y_bottom - val * (y_bottom - y_top)

    d.line([mx, y0 + 12, mx, y1 - 18], fill=(*GREEN, 135), width=1)
    d.ellipse([mx - 6, my - 6, mx + 6, my + 6], fill=(*WHITE, 230), outline=(*GREEN, 230), width=1)

    d.text((x0 + 18, y0 + 12), "PULSE PROFILE", font=font, fill=(*CYAN2, 175))
    d.text((x1 - 150, y0 + 12), "PHASE", font=font, fill=(*GREEN, 150))


def draw_telemetry_box(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    rows: list[tuple[str, str]],
    font,
):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 150))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    y = y0 + 18

    for key, value in rows:
        d.text((x0 + 18, y), key, font=font, fill=(*CYAN, 150))
        d.text((x0 + 148, y), value, font=font, fill=(*CYAN2, 215))
        y += 30


def draw_pulsar_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(22)
    font_small = load_font(15)
    font_mid = load_font(18)

    W, H = frame.size
    phase = 2 * np.pi * i / TOTAL_FRAMES

    draw_hud_frame(frame, font_big)
    draw_grid(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(scene)

    cx = W * 0.43
    cy = H * 0.39
    star_r = 44

    observer_angle = np.deg2rad(16)
    intensity = min(1.0, pulse_intensity(phase, observer_angle))

    # observer line
    obs_len = 360
    obs_x = cx + obs_len * np.cos(observer_angle)
    obs_y = cy + obs_len * np.sin(observer_angle)

    d.line([cx, cy, obs_x, obs_y], fill=(*GREEN, 95), width=1)
    d.text((obs_x - 30, obs_y + 10), "OBSERVER", font=font_small, fill=(*GREEN, 170))

    # rotation axis
    axis_angle = np.deg2rad(-82)
    axis_len = 170
    d.line(
        [
            cx - axis_len * np.cos(axis_angle),
            cy - axis_len * np.sin(axis_angle),
            cx + axis_len * np.cos(axis_angle),
            cy + axis_len * np.sin(axis_angle),
        ],
        fill=(*CYAN, 145),
        width=2,
    )
    d.text(
        (cx + axis_len * np.cos(axis_angle) + 8, cy + axis_len * np.sin(axis_angle) - 8),
        "ROT AXIS",
        font=font_small,
        fill=(*CYAN, 170),
    )

    # magnetic beam axis rotates
    beam_angle = phase + np.deg2rad(14)
    beam_len = 360
    beam_alpha = int(40 + 130 * intensity)

    draw_beam_cone(
        scene,
        (cx, cy),
        beam_angle,
        beam_len,
        spread=np.deg2rad(10),
        color=YELLOW,
        alpha=beam_alpha,
    )

    draw_beam_cone(
        scene,
        (cx, cy),
        beam_angle + np.pi,
        beam_len * 0.88,
        spread=np.deg2rad(10),
        color=YELLOW,
        alpha=int(beam_alpha * 0.72),
    )

    # magnetic axis line
    d.line(
        [
            cx - beam_len * 0.32 * np.cos(beam_angle),
            cy - beam_len * 0.32 * np.sin(beam_angle),
            cx + beam_len * 0.42 * np.cos(beam_angle),
            cy + beam_len * 0.42 * np.sin(beam_angle),
        ],
        fill=(*YELLOW, 190),
        width=2,
    )

    draw_neutron_star(scene, cx, cy, star_r, phase)

    # pulse flash overlay
    if intensity > 0.25:
        flash_r = 58 + 26 * intensity
        d.ellipse(
            [cx - flash_r, cy - flash_r, cx + flash_r, cy + flash_r],
            outline=(*WHITE, int(210 * intensity)),
            width=3,
        )

    d.text((82, 112), "ROTATING MAGNETIC DIPOLE / LIGHTHOUSE MODEL", font=font_mid, fill=(*CYAN2, 210))
    d.text((82, 140), "OBSERVED PULSE OCCURS WHEN BEAM INTERSECTS LINE OF SIGHT", font=font_small, fill=(*CYAN2, 155))

    rows = [
        ("period", "33.4 ms"),
        ("B-field", "1.2e12 G"),
        ("beam", f"{np.degrees(beam_angle) % 360:06.2f} deg"),
        ("pulse", f"{intensity:.3f}"),
        ("state", "DETECTED" if intensity > 0.42 else "SCAN"),
    ]

    draw_telemetry_box(
        scene,
        (W - 365, 112, W - 84, 286),
        rows,
        font_small,
    )

    draw_pulse_profile(
        scene,
        (90, H - 210, W - 90, H - 82),
        phase,
        observer_angle,
        font_small,
    )

    # phase ring
    ring_cx, ring_cy = W - 120, H - 298
    ring_r = 42
    d.ellipse(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        outline=(*CYAN, 70),
        width=1,
    )
    d.arc(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        start=-90,
        end=-90 + 360 * i / TOTAL_FRAMES,
        fill=(*CYAN2, 220),
        width=4,
    )
    d.text((ring_cx - 42, ring_cy + 54), "ROT PHASE", font=font_small, fill=(*CYAN2, 145))

    glow_composite(frame, scene, blur=5)

    return frame


def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] pulsar beam telemetry")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_pulsar_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(
        f"Created 1 animation(s) as '{output_format}' "
        f"in '{ANIMATIONS_DIR.resolve()}'"
    )


main()

[START] pulsar beam telemetry
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] telemetry_pulsar_beam
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_pulsar_beam/telemetry_pulsar_beam.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[out#0/webm @ 0x152e262b0] video:398KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 89.422978%
frame=  180 fps= 34 q=34.0 Lsize=     755KiB time=00:00:07.50 bitrate= 824.2kbits/s speed=1.42x    


In [18]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 180

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "telemetry_gravitational_lensing"

CANVAS_SIZE = (1200, 680)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)


# =========================================================
# BASIC HELPERS
# =========================================================

def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be: webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))

    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 5):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def bezier(p0, p1, p2, p3, n=120):
    pts = []

    for t in np.linspace(0, 1, n):
        p = (
            (1 - t) ** 3 * np.array(p0)
            + 3 * (1 - t) ** 2 * t * np.array(p1)
            + 3 * (1 - t) * t ** 2 * np.array(p2)
            + t ** 3 * np.array(p3)
        )
        pts.append(tuple(p))

    return pts


# =========================================================
# HUD
# =========================================================

def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 34
    cut = 52

    pts = [
        (pad + cut, pad),
        (W - pad, pad),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 22, pad + 22, W - pad - 22, H - pad - 22],
        outline=(*CYAN2, 42),
        width=1,
    )

    d.text(
        (pad + 24, pad + 14),
        "TELEMETRY // GRAVITATIONAL LENSING",
        font=font,
        fill=(*CYAN2, 220),
    )


def draw_grid(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 86, x, H - 70], fill=(*CYAN, 13), width=1)

    for y in range(100, H - 70, 60):
        d.line([84, y, W - 84, y], fill=(*CYAN, 11), width=1)


# =========================================================
# SCENE PARTS
# =========================================================

def draw_source(layer: Image.Image, cx: float, cy: float, phase: float, font):
    d = ImageDraw.Draw(layer)

    r = 18 + 4 * np.sin(phase * 2) ** 2

    d.ellipse(
        [cx - r * 2.2, cy - r * 1.2, cx + r * 2.2, cy + r * 1.2],
        outline=(*CYAN2, 90),
        width=1,
    )

    d.ellipse(
        [cx - r, cy - r, cx + r, cy + r],
        fill=(*CYAN2, 150),
        outline=(*WHITE, 190),
        width=1,
    )

    d.text(
        (cx - 42, cy + 30),
        "SOURCE",
        font=font,
        fill=(*CYAN2, 180),
    )


def draw_lens_body(layer: Image.Image, cx: float, cy: float, alignment: float, font):
    d = ImageDraw.Draw(layer)

    halo_base = 124

    for rr, alpha in [
        (halo_base, 12),
        (halo_base * 0.72, 26),
        (halo_base * 0.46, 48),
    ]:
        d.ellipse(
            [cx - rr, cy - rr, cx + rr, cy + rr],
            fill=(*PURPLE, int(alpha * (0.7 + 0.3 * alignment))),
        )

    # black hole / compact mass
    d.ellipse(
        [cx - 43, cy - 43, cx + 43, cy + 43],
        fill=(*BG_DARK, 250),
        outline=(*PURPLE, 235),
        width=3,
    )

    d.ellipse(
        [cx - 17, cy - 17, cx + 17, cy + 17],
        fill=(*PURPLE, 170),
        outline=(*WHITE, 90),
        width=1,
    )

    d.text(
        (cx + 52, cy - 10),
        "LENS MASS",
        font=font,
        fill=(*PURPLE, 220),
    )


def draw_einstein_ring(layer: Image.Image, cx: float, cy: float, phase: float, alignment: float):
    d = ImageDraw.Draw(layer)

    base_r = 94
    ring_r = base_r * (0.82 + 0.22 * alignment)
    arc_alpha = int(55 + 180 * alignment)

    d.ellipse(
        [cx - ring_r, cy - ring_r, cx + ring_r, cy + ring_r],
        outline=(*CYAN2, int(38 + 58 * alignment)),
        width=1,
    )

    for k in range(4):
        start = np.degrees(phase * 0.7 + k * np.pi / 2)
        extent = 42 + 58 * alignment
        color = [CYAN2, GREEN, YELLOW, CYAN][k % 4]

        d.arc(
            [cx - ring_r, cy - ring_r, cx + ring_r, cy + ring_r],
            start=start,
            end=start + extent,
            fill=(*color, arc_alpha),
            width=5,
        )


def draw_light_rays(
    layer: Image.Image,
    source: tuple[float, float],
    lens: tuple[float, float],
    observer: tuple[float, float],
    alignment: float,
):
    sx, sy = source
    lx, ly = lens
    ox, oy = observer

    ray_layer = Image.new("RGBA", layer.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(ray_layer)

    R = 92 + 10 * alignment
    theta = 0.50

    ray_specs = [
        (-8, -1, CYAN2),
        (-3, -1, WHITE),
        (3, 1, WHITE),
        (8, 1, GREEN),
    ]

    for src_off, side, color in ray_specs:
        if side < 0:
            a0 = np.pi + theta
            a1 = -theta
            arc_angles = np.linspace(a0, a1, 92)
        else:
            a0 = np.pi - theta
            a1 = theta
            arc_angles = np.linspace(a0, a1, 92)

        p_left = (
            lx + R * np.cos(a0),
            ly + R * np.sin(a0),
        )

        p_right = (
            lx + R * np.cos(a1),
            ly + R * np.sin(a1),
        )

        p0 = (sx, sy + src_off)
        p_end = (ox, oy + src_off * 0.35)

        first = bezier(
            p0,
            (sx + 145, sy + src_off),
            (lx - 145, p_left[1]),
            p_left,
            n=75,
        )

        arc = [
            (
                lx + R * np.cos(a),
                ly + R * np.sin(a),
            )
            for a in arc_angles
        ]

        second = bezier(
            p_right,
            (lx + 145, p_right[1]),
            (ox - 145, oy + src_off * 0.35),
            p_end,
            n=75,
        )

        pts = first + arc + second

        # thick visible beam
        d.line(pts, fill=(*color, 245), width=4)

        # hot core line
        d.line(pts, fill=(*WHITE, 155), width=1)

    glow = ray_layer.filter(ImageFilter.GaussianBlur(5))
    layer.alpha_composite(glow)
    layer.alpha_composite(ray_layer)


def draw_observer(layer: Image.Image, x: float, y: float, font):
    d = ImageDraw.Draw(layer)

    d.line(
        [x, y - 82, x, y + 82],
        fill=(*GREEN, 185),
        width=2,
    )

    d.rectangle(
        [x - 11, y - 60, x + 11, y + 60],
        outline=(*GREEN, 125),
        width=1,
    )

    d.text(
        (x + 18, y - 10),
        "OBSERVER",
        font=font,
        fill=(*GREEN, 185),
    )


def draw_magnification_curve(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    progress: float,
    font,
):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 150))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    for gx in np.linspace(x0 + 20, x1 - 20, 8):
        d.line([gx, y0 + 14, gx, y1 - 18], fill=(*CYAN, 26), width=1)

    for gy in np.linspace(y0 + 16, y1 - 24, 5):
        d.line([x0 + 14, gy, x1 - 14, gy], fill=(*CYAN, 22), width=1)

    xs = np.linspace(0, 1, 480)
    pts = []

    y_top = y0 + 24
    y_bottom = y1 - 30

    def mag_curve(t):
        val = np.exp(-0.5 * ((t - 0.5) / 0.12) ** 2)
        val += 0.18 * np.exp(-0.5 * ((t - 0.22) / 0.07) ** 2)
        val += 0.18 * np.exp(-0.5 * ((t - 0.78) / 0.07) ** 2)
        return min(1.0, val)

    for t in xs:
        val = mag_curve(t)
        x = x0 + 18 + t * (x1 - x0 - 36)
        y = y_bottom - val * (y_bottom - y_top)
        pts.append((x, y))

    d.line(pts, fill=(*CYAN2, 230), width=2)

    mx = x0 + 18 + progress * (x1 - x0 - 36)
    val = mag_curve(progress)
    my = y_bottom - val * (y_bottom - y_top)

    d.line([mx, y0 + 12, mx, y1 - 18], fill=(*GREEN, 135), width=1)
    d.ellipse(
        [mx - 6, my - 6, mx + 6, my + 6],
        fill=(*WHITE, 230),
        outline=(*GREEN, 230),
        width=1,
    )

    d.text(
        (x0 + 18, y0 + 12),
        "MAGNIFICATION CURVE",
        font=font,
        fill=(*CYAN2, 175),
    )

    d.text(
        (x1 - 145, y0 + 12),
        "ALIGNMENT",
        font=font,
        fill=(*GREEN, 150),
    )


def draw_telemetry_box(
    layer: Image.Image,
    box: tuple[int, int, int, int],
    rows: list[tuple[str, str]],
    font,
):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 150))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    y = y0 + 18

    for key, value in rows:
        d.text((x0 + 18, y), key, font=font, fill=(*CYAN, 150))
        d.text((x0 + 155, y), value, font=font, fill=(*CYAN2, 215))
        y += 30


# =========================================================
# MAIN SCENE
# =========================================================

def draw_lensing_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(22)
    font_small = load_font(15)
    font_mid = load_font(18)

    W, H = frame.size

    progress = i / TOTAL_FRAMES
    phase = 2 * np.pi * progress

    draw_hud_frame(frame, font_big)
    draw_grid(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(scene)

    source_x = W * 0.18
    lens_x = W * 0.48
    observer_x = W * 0.76

    cy = H * 0.39

    source_y_bottom = cy + 105
    source_y_top = cy - 105
    source_y = source_y_bottom + (source_y_top - source_y_bottom) * progress

    lens_y = cy
    observer_y = cy

    alignment = float(np.exp(-0.5 * ((source_y - lens_y) / 44) ** 2))

    # Static guide / optical axis
    d.line(
        [source_x - 40, cy, observer_x + 60, cy],
        fill=(*CYAN, 38),
        width=1,
    )

    draw_source(scene, source_x, source_y, phase, font_small)

    draw_lens_body(scene, lens_x, lens_y, alignment, font_small)

    draw_einstein_ring(scene, lens_x, lens_y, phase, alignment)

    # top overlay compact mass
    d.ellipse(
        [lens_x - 35, lens_y - 35, lens_x + 35, lens_y + 35],
        fill=(*BG_DARK, 245),
        outline=(*PURPLE, 245),
        width=2,
    )

    d.ellipse(
        [lens_x - 12, lens_y - 12, lens_x + 12, lens_y + 12],
        fill=(*PURPLE, 190),
    )

    # Rays deliberately drawn over lens/ring so they remain visible
    draw_light_rays(
        scene,
        source=(source_x, source_y),
        lens=(lens_x, lens_y),
        observer=(observer_x, observer_y),
        alignment=alignment,
    )

    draw_observer(scene, observer_x, observer_y, font_small)

    d.text(
        (82, 112),
        "BACKGROUND SOURCE / FOREGROUND MASS / DISTORTED LIGHT PATHS",
        font=font_mid,
        fill=(*CYAN2, 210),
    )

    d.text(
        (82, 140),
        "SOURCE SLIDES THROUGH ALIGNMENT: ARCS INTENSIFY NEAR OPTICAL AXIS",
        font=font_small,
        fill=(*CYAN2, 155),
    )

    theta_e = 1.12 + 0.34 * alignment
    mag = 1.0 + 3.8 * alignment

    rows = [
        ("lens mass", "4.2e11 Msun"),
        ("theta_E", f"{theta_e:.2f} arcsec"),
        ("align", f"{alignment:.3f}"),
        ("magnif.", f"{mag:.2f}x"),
        ("mode", "EINSTEIN RING" if alignment > 0.72 else "ARC"),
    ]

    draw_telemetry_box(
        scene,
        (W - 345, 78, W - 58, 252),
        rows,
        font_small,
    )

    draw_magnification_curve(
        scene,
        (90, H - 210, W - 90, H - 82),
        progress,
        font_small,
    )

    ring_cx, ring_cy = W - 120, H - 298
    ring_r = 42

    d.ellipse(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        outline=(*CYAN, 70),
        width=1,
    )

    d.arc(
        [ring_cx - ring_r, ring_cy - ring_r, ring_cx + ring_r, ring_cy + ring_r],
        start=-90,
        end=-90 + 360 * progress,
        fill=(*CYAN2, 220),
        width=4,
    )

    d.text(
        (ring_cx - 48, ring_cy + 54),
        "LENSING PHASE",
        font=font_small,
        fill=(*CYAN2, 145),
    )

    glow_composite(frame, scene, blur=5)

    return frame


# =========================================================
# EXPORT
# =========================================================

def export_webm(
    frames: list[Image.Image],
    out_path: Path,
    fps: int,
    keep_frames: bool = False,
) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-framerate",
            str(fps),
            "-i",
            str(frame_dir / "frame_%04d.png"),
            "-c:v",
            "libvpx-vp9",
            "-b:v",
            "0",
            "-crf",
            "34",
            "-pix_fmt",
            "yuva420p",
            "-auto-alt-ref",
            "0",
            "-row-mt",
            "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(
    frames: list[Image.Image],
    out_path: Path,
    fps: int,
    keep_frames: bool = False,
) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-framerate",
            str(fps),
            "-i",
            str(frame_dir / "frame_%04d.png"),
            "-c:v",
            "libx264",
            "-crf",
            "22",
            "-pix_fmt",
            "yuv420p",
            "-movflags",
            "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


# =========================================================
# RUN
# =========================================================

def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] gravitational lensing telemetry")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_lensing_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(
        f"Created 1 animation(s) as '{output_format}' "
        f"in '{ANIMATIONS_DIR.resolve()}'"
    )


main()

[START] gravitational lensing telemetry
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] telemetry_gravitational_lensing
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_gravitational_lensing/telemetry_gravitational_lensing.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[out#0/webm @ 0x15c00bcf0] video:373KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 81.347739%
frame=  180 fps= 36 q=34.0 Lsize=     677KiB time=00:00:07.50 bitrate= 739.5kbits/s speed=1.49x    


In [19]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont


# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 180

ANIMATIONS_DIR = Path("media-site/animations")
ANIMATION_NAME = "telemetry_hr_diagram"

CANVAS_SIZE = (1200, 680)

BG_DARK = (2, 7, 13)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)


# =========================================================
# HELPERS
# =========================================================

def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT must be: webm, gif or mp4")
    return fmt


def load_font(size: int):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def make_canvas(output_format: str) -> Image.Image:
    if normalize_output_format(output_format) == "webm":
        return Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    return Image.new("RGBA", CANVAS_SIZE, (*BG_DARK, 255))


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


def glow_composite(base: Image.Image, layer: Image.Image, blur: int = 5):
    glow = layer.filter(ImageFilter.GaussianBlur(blur))
    base.alpha_composite(glow)
    base.alpha_composite(layer)


def smoothstep(t: float) -> float:
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def lerp(a, b, t):
    return a + (b - a) * t


# =========================================================
# HUD
# =========================================================

def draw_hud_frame(img: Image.Image, font):
    d = ImageDraw.Draw(img)
    W, H = img.size

    pad = 34
    cut = 52

    pts = [
        (pad + cut, pad),
        (W - pad, pad),
        (W - pad, H - pad - cut),
        (W - pad - cut, H - pad),
        (pad, H - pad),
        (pad, pad + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 135), width=2)

    d.rectangle(
        [pad + 22, pad + 22, W - pad - 22, H - pad - 22],
        outline=(*CYAN2, 42),
        width=1,
    )

    d.text(
        (pad + 24, pad + 14),
        "TELEMETRY // HERTZSPRUNG-RUSSELL DIAGRAM",
        font=font,
        fill=(*CYAN2, 220),
    )


def draw_grid_background(img: Image.Image):
    d = ImageDraw.Draw(img)
    W, H = img.size

    for x in range(80, W - 80, 80):
        d.line([x, 86, x, H - 70], fill=(*CYAN, 12), width=1)

    for y in range(100, H - 70, 60):
        d.line([84, y, W - 84, y], fill=(*CYAN, 10), width=1)


# =========================================================
# HR DIAGRAM
# =========================================================

def hr_box():
    return 110, 116, 820, 540


def hr_map(log_temp: float, log_lum: float) -> tuple[float, float]:
    """
    HR convention:
    hot stars on the left, cool stars on the right.
    x: log10(T_eff), range 3.35..4.65
    y: log10(L/Lsun), range -4..6
    """
    x0, y0, x1, y1 = hr_box()

    t_min, t_max = 3.35, 4.65
    l_min, l_max = -4.0, 6.0

    x = x0 + (t_max - log_temp) / (t_max - t_min) * (x1 - x0)
    y = y1 - (log_lum - l_min) / (l_max - l_min) * (y1 - y0)

    return x, y


def spectral_color(log_temp: float) -> tuple[int, int, int]:
    if log_temp > 4.45:
        return (140, 190, 255)
    if log_temp > 4.15:
        return (190, 220, 255)
    if log_temp > 3.95:
        return WHITE
    if log_temp > 3.78:
        return (255, 245, 170)
    if log_temp > 3.60:
        return (255, 205, 110)
    return (255, 120, 90)


def generate_star_catalog(seed: int = 42, n_main: int = 420, n_giants: int = 110, n_wd: int = 95):
    rng = np.random.default_rng(seed)
    stars = []

    # Main sequence: diagonal band
    for _ in range(n_main):
        u = rng.random()
        log_t = lerp(4.55, 3.45, u) + rng.normal(0, 0.035)
        log_l = lerp(5.2, -2.7, u) + rng.normal(0, 0.32)
        stars.append(("main", log_t, log_l, rng.uniform(1.2, 3.0)))

    # Giants / supergiants
    for _ in range(n_giants):
        log_t = rng.uniform(3.48, 3.86)
        log_l = rng.uniform(1.3, 4.7) + rng.normal(0, 0.15)
        stars.append(("giant", log_t, log_l, rng.uniform(1.8, 4.2)))

    # White dwarfs
    for _ in range(n_wd):
        log_t = rng.uniform(3.85, 4.55)
        log_l = rng.uniform(-3.4, -1.0) + rng.normal(0, 0.12)
        stars.append(("wd", log_t, log_l, rng.uniform(1.1, 2.5)))

    rng.shuffle(stars)
    return stars


STAR_CATALOG = generate_star_catalog()


def draw_hr_axes(layer: Image.Image, font_small, font_tiny):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 135))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 105), width=1)

    # vertical temperature ticks
    temp_ticks = [
        (4.6, "40000"),
        (4.3, "20000"),
        (4.0, "10000"),
        (3.8, "6300"),
        (3.6, "4000"),
        (3.4, "2500"),
    ]

    for log_t, label in temp_ticks:
        x, _ = hr_map(log_t, -4)
        d.line([x, y0, x, y1], fill=(*CYAN, 25), width=1)
        d.text((x - 24, y1 + 12), label, font=font_tiny, fill=(*CYAN2, 145))

    # horizontal luminosity ticks
    lum_ticks = [6, 4, 2, 0, -2, -4]
    for log_l in lum_ticks:
        _, y = hr_map(3.35, log_l)
        d.line([x0, y, x1, y], fill=(*CYAN, 24), width=1)
        d.text((x0 - 54, y - 8), f"10^{log_l}", font=font_tiny, fill=(*CYAN2, 145))

    d.text((x0, y1 + 40), "TEMPERATURE K  ← HOTTER", font=font_small, fill=(*CYAN2, 180))
    d.text((x0 - 68, y0 - 28), "LUMINOSITY", font=font_small, fill=(*CYAN2, 180))

    # spectral classes
    classes = [
        ("O", 4.55, (140, 190, 255)),
        ("B", 4.30, (180, 210, 255)),
        ("A", 4.08, WHITE),
        ("F", 3.90, (255, 250, 190)),
        ("G", 3.76, (255, 225, 120)),
        ("K", 3.62, (255, 175, 90)),
        ("M", 3.45, (255, 105, 80)),
    ]

    for label, log_t, col in classes:
        x, _ = hr_map(log_t, -4)
        d.text((x - 6, y1 + 66), label, font=font_small, fill=(*col, 210))


def draw_hr_regions(layer: Image.Image, font_tiny, phase: float):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = hr_box()

    pulse = 0.6 + 0.4 * np.sin(phase * 2) ** 2

    # Main sequence highlight as slanted polyline band
    band = []
    for u in np.linspace(0, 1, 80):
        log_t = lerp(4.55, 3.45, u)
        log_l = lerp(5.2, -2.7, u)
        band.append(hr_map(log_t, log_l))

    d.line(band, fill=(*CYAN2, int(80 + 45 * pulse)), width=18)
    d.line(band, fill=(*CYAN2, 120), width=2)

    # Giant region
    gx0, gy0 = hr_map(3.88, 4.8)
    gx1, gy1 = hr_map(3.42, 1.0)
    d.rectangle([gx0, gy0, gx1, gy1], outline=(*ORANGE, 80), width=1)
    d.text((gx0 + 8, gy0 + 8), "GIANTS", font=font_tiny, fill=(*ORANGE, 165))

    # White dwarf region
    wx0, wy0 = hr_map(4.55, -0.8)
    wx1, wy1 = hr_map(3.82, -3.8)
    d.rectangle([wx0, wy0, wx1, wy1], outline=(*PURPLE, 85), width=1)
    d.text((wx0 + 8, wy1 - 24), "WHITE DWARFS", font=font_tiny, fill=(*PURPLE, 170))

    d.text((x0 + 370, y0 + 260), "MAIN SEQUENCE", font=font_tiny, fill=(*CYAN2, 170))


def draw_star_points(layer: Image.Image, progress: float):
    d = ImageDraw.Draw(layer)

    n_visible = int(len(STAR_CATALOG) * smoothstep(progress))

    for idx, (_kind, log_t, log_l, radius) in enumerate(STAR_CATALOG[:n_visible]):
        x, y = hr_map(log_t, log_l)
        col = spectral_color(log_t)

        alpha = 75
        if idx > n_visible - 30:
            alpha = 55 + int(145 * (idx - max(0, n_visible - 30)) / 30)

        d.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            fill=(*col, alpha),
        )


def draw_target_track(layer: Image.Image, phase: float, font_small):
    d = ImageDraw.Draw(layer)

    # closed loop track: moves from hot/luminous to WD region and back subtly
    u = (0.5 + 0.5 * np.sin(phase - np.pi / 2))

    log_t = lerp(4.35, 4.05, u)
    log_l = lerp(1.2, -2.2, u)

    x, y = hr_map(log_t, log_l)

    pulse = 0.55 + 0.45 * np.sin(phase * 4) ** 2

    d.ellipse(
        [x - 18, y - 18, x + 18, y + 18],
        outline=(*GREEN, int(130 + 90 * pulse)),
        width=2,
    )

    d.line([x - 26, y, x + 26, y], fill=(*GREEN, 160), width=1)
    d.line([x, y - 26, x, y + 26], fill=(*GREEN, 160), width=1)

    d.ellipse(
        [x - 5, y - 5, x + 5, y + 5],
        fill=(*WHITE, 230),
    )

    d.text((x + 22, y - 10), "TARGET", font=font_small, fill=(*GREEN, 220))

    return log_t, log_l


def draw_telemetry_box(layer: Image.Image, box: tuple[int, int, int, int], rows: list[tuple[str, str]], font):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 150))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    y = y0 + 18

    for key, value in rows:
        d.text((x0 + 18, y), key, font=font, fill=(*CYAN, 150))
        d.text((x0 + 145, y), value, font=font, fill=(*CYAN2, 215))
        y += 30


def draw_population_meter(layer: Image.Image, box: tuple[int, int, int, int], progress: float, font):
    d = ImageDraw.Draw(layer)
    x0, y0, x1, y1 = box

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 145))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 80), width=1)

    d.text((x0 + 14, y0 + 12), "CATALOG POPULATION", font=font, fill=(*CYAN2, 170))

    bar_x0 = x0 + 16
    bar_y0 = y0 + 48
    bar_x1 = x1 - 16
    bar_y1 = y0 + 70

    d.rectangle([bar_x0, bar_y0, bar_x1, bar_y1], outline=(*CYAN, 80), width=1)

    fill_x = bar_x0 + (bar_x1 - bar_x0) * smoothstep(progress)

    d.rectangle([bar_x0, bar_y0, fill_x, bar_y1], fill=(*CYAN2, 150))

    count = int(len(STAR_CATALOG) * smoothstep(progress))
    d.text((bar_x0, bar_y1 + 16), f"{count:04d} OBJECTS", font=font, fill=(*GREEN, 180))


def draw_hr_scene(i: int, output_format: str) -> Image.Image:
    frame = make_canvas(output_format)

    font_big = load_font(22)
    font_mid = load_font(18)
    font_small = load_font(15)
    font_tiny = load_font(13)

    W, H = frame.size

    phase = 2 * np.pi * i / TOTAL_FRAMES
    progress = i / TOTAL_FRAMES

    draw_hud_frame(frame, font_big)
    draw_grid_background(frame)

    scene = Image.new("RGBA", frame.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(scene)

    draw_hr_axes(scene, font_small, font_tiny)
    draw_hr_regions(scene, font_tiny, phase)
    draw_star_points(scene, progress)

    log_t, log_l = draw_target_track(scene, phase, font_small)

    temp = 10 ** log_t
    lum = 10 ** log_l

    if log_l < -1.0:
        stage = "WHITE DWARF TRACK"
        sclass = "DA / HOT WD"
    elif log_l > 1.0:
        stage = "GIANT BRANCH"
        sclass = "K / M GIANT"
    else:
        stage = "MAIN SEQUENCE"
        sclass = "A / F"

    rows = [
        ("T_eff", f"{temp:,.0f} K"),
        ("log L", f"{log_l:+.2f}"),
        ("L/Lsun", f"{lum:.3f}"),
        ("class", sclass),
        ("stage", stage),
    ]

    draw_telemetry_box(
        scene,
        (875, 116, 1120, 296),
        rows,
        font_small,
    )

    draw_population_meter(
        scene,
        (875, 330, 1120, 432),
        progress,
        font_small,
    )

    d.text(
        (110, 82),
        "STELLAR POPULATION MAP / TEMPERATURE VS LUMINOSITY",
        font=font_mid,
        fill=(*CYAN2, 210),
    )

    d.text(
        (875, 470),
        "REGIONS:",
        font=font_small,
        fill=(*CYAN2, 170),
    )

    legend = [
        ("MAIN SEQUENCE", CYAN2),
        ("GIANTS", ORANGE),
        ("WHITE DWARFS", PURPLE),
        ("TARGET TRACK", GREEN),
    ]

    yy = 498
    for label, col in legend:
        d.rectangle([875, yy + 4, 888, yy + 17], fill=(*col, 150))
        d.text((898, yy), label, font=font_small, fill=(*col, 185))
        yy += 28

    glow_composite(frame, scene, blur=4)

    return frame


# =========================================================
# EXPORT
# =========================================================

def export_webm(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libvpx-vp9",
            "-b:v", "0",
            "-crf", "34",
            "-pix_fmt", "yuva420p",
            "-auto-alt-ref", "0",
            "-row-mt", "1",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_mp4(frames: list[Image.Image], out_path: Path, fps: int, keep_frames: bool = False) -> Path:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    frame_dir = out_path.parent / "_frames"
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", str(frame_dir / "frame_%04d.png"),
            "-c:v", "libx264",
            "-crf", "22",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(out_path),
        ],
        check=True,
    )

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path


def export_gif(frames: list[Image.Image], out_path: Path, fps: int) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(frames: list[Image.Image], output_format: str) -> Path:
    output_format = normalize_output_format(output_format)

    folder = ANIMATIONS_DIR / ANIMATION_NAME
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{ANIMATION_NAME}.{output_format}"

    if output_format == "webm":
        return export_webm(frames, out_path, FPS)

    if output_format == "mp4":
        return export_mp4(frames, out_path, FPS)

    return export_gif(frames, out_path, FPS)


# =========================================================
# RUN
# =========================================================

def main():
    output_format = normalize_output_format(OUTPUT_FORMAT)

    print("[START] HR diagram telemetry")
    print(f"[CONFIG] OUTPUT_FORMAT = {output_format}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    frames = []

    for i in range(TOTAL_FRAMES):
        if i % 24 == 0:
            print(f"[GENERATE] frame {i}/{TOTAL_FRAMES}")

        frames.append(draw_hr_scene(i, output_format))

    saved_path = export_animation(frames, output_format)

    print()
    print(f"[CREATED] {ANIMATION_NAME}")
    print(f"          {saved_path.resolve()}")
    print()
    print(
        f"Created 1 animation(s) as '{output_format}' "
        f"in '{ANIMATIONS_DIR.resolve()}'"
    )


main()

[START] HR diagram telemetry
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 180
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations
[GENERATE] frame 0/180
[GENERATE] frame 24/180
[GENERATE] frame 48/180
[GENERATE] frame 72/180
[GENERATE] frame 96/180
[GENERATE] frame 120/180
[GENERATE] frame 144/180
[GENERATE] frame 168/180


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib


[CREATED] telemetry_hr_diagram
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/telemetry_hr_diagram/telemetry_hr_diagram.webm

Created 1 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[out#0/webm @ 0x11d0050b0] video:246KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 64.643537%
frame=  180 fps= 39 q=34.0 Lsize=     405KiB time=00:00:07.50 bitrate= 442.9kbits/s speed=1.62x    
